# ZTE Parallax — three vantage points on the same minds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/feature/decoder/notebooks/zte_parallax.ipynb)

**The confound, and the structural answer.** ZuCo's task is fully confounded with its stimulus set: Cramér's
$V(\text{task}, \text{stimulus}) = 0.998$, and no sentence appears under two tasks. A model trained across tasks can
therefore win its contrastive game on task register rather than content — and the measured encoder does exactly
that, *amplifying* the task probe to 0.918 from a raw-feature 0.685. Parallax removes the confound structurally
instead of adversarially: **three independent encoders, one per task (NR, SR, TSR)**, each trained only on its own
task's readings with the current best-measured recipe (residual coding off, gallery CE off — both falsified by
their matched ablations on 2026-08-15). Within one task there is no task variable left to lean on.

**The prize.** The scientific readout is the **3 × 3 cross-task transfer matrix**. A model trained on task X and
evaluated on task Y ≠ X faces a never-seen subject (LOSO holdout `ZAB`) reading stimuli disjoint from everything the
model was trained on — the strongest generalisation cell this project can produce. The honest expectation: the
in-task diagonal well above chance, the cross-task cells possibly null. **A null is a finding** and is reported
plainly. The metaphor is parallax — three vantage points observing the same minds; what stays fixed across vantage
points, measured by transfer and by CKA between the three models, is the task-invariant thought code.

**The readout is closed-set.** Stratified retrieval rank percentile with a bootstrap CI, plus the menu-capacity
audit over exact-length pools — within one task the task dimension is constant by construction, and exact-length
pools remove ZuCo's free 5.14 bits of sentence length. **Free generation is not a parallax deliverable**: the
encoder's bit budget prices it out, and no generation number appears in this study. Phase 3 (§9) trains a decoder
over the winning encoder, and its generation readout is the pre-registered expected null.

| You want to | Go to |
| --- | --- |
| Set up a fresh Colab runtime | §1 – §3 |
| Train the three vantage points | §4 |
| Fill the 3 × 3 transfer matrix | §5 |
| Build the report and walk the chamber | §6 |
| Read the outcome honestly | §7 |
| Inspect one reading through the lens | §8 |
| Train the decoder over the winner (Phase 3) | §9 |

## 1 · Provision the runtime

Colab ships an older Python than ZTE requires (`>=3.14`), so `uv` provisions the pinned interpreter and installs
everything into a cached virtualenv. The clone is shallow and hard-resets to the branch, so only git-tracked files
move — your Drive folder and any local cache are untouched. Re-running this cell on a warm runtime is fast and
idempotent.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch feature/decoder && cd zte
fi
git fetch --depth 1 origin feature/decoder && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

Colab's kernel is an **older interpreter than the 3.14 ZTE requires**, so `import zte` here is a `SyntaxError`. It
never needs to: every capability arrives through `zte-colab`, one subcommand per question, each printing a single
JSON object on stdout with its logs on stderr. `colab()` below is that bridge and the only way this notebook
reaches the package — everything else in a code cell is the standard library, `IPython.display`, `google.colab`,
and Colab's own renderers.

In [ ]:
import json
import os
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) — Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded — authenticated HuggingFace Hub downloads enabled.')

print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} · zte {ENV["venv"]["zte"]}   ← every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   ← this cell; renders payloads, never imports zte')

## 3 · Drive is the workspace

A Colab VM can vanish without warning, so everything durable lives on Drive: one shared folder for data plus one
dated folder per session.

```text
Sharables/ZTE/
├── prepared/       # cached feature bundles, NOT date-stamped: built once, reused forever
├── ZuCo Dataset/   # the raw .mat archives — task1 (SR), task2 (NR), task3 (TSR)
└── YYYY-MM-DD/     # one folder per session: experiments/, analysis/, archives/
```

Training checkpoints go to the VM's fast local disk and are mirrored to Drive after every run, because a Drive FUSE
stall mid-`torch.save` is a torn checkpoint. Everything else — the transfer cells' aggregate report, the chamber —
is written straight to Drive. To resume an interrupted session, set `RESUME_DATE` to that session's folder name;
every `--resume` then finds its work already done.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
# Set to an existing folder name (e.g. '2026-08-15') to resume that session; None starts today's.
RESUME_DATE: str | None = None
# 'local+mirror' trains on the VM disk and copies to Drive after each run (recommended).
WRITE_MODE: str = 'local+mirror'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *_resume)

# Every `!` command below inherits these, so the bundle cache, the data root and the backup target are wired once.
os.environ.update(SESSION['env'])

RUN_DATE: str = SESSION['run_date']
DATA_DIR: str = SESSION['data_dir']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
OUT_ROOT: str = SESSION['out_root']
DRIVE_BACKUP: str = SESSION['drive_backup']
PREPARED_LOCAL: str = SESSION['prepared_local']
PREPARED_DRIVE: str = SESSION['prepared_drive']

print(f'session   : {RUN_DATE}   ({"resumed" if SESSION["resumed"] else "new"})')
print(f'Drive     : {SESSION["drive_root"]}   (mounted: {SESSION["drive_mounted"]})')
print(f'raw data  : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'runs ->   : {OUT_ROOT}   (backed up to {DRIVE_BACKUP})')
print(f'analysis  : {DRIVE_ANALYSIS}')
print(f'prepared  : {PREPARED_DRIVE}   (staged on the VM at {PREPARED_LOCAL})')

### 3a · Helpers this notebook uses everywhere

Small and boring: how runs move between the VM and Drive in both directions, plus the resource readout. Each is a
thin renderer over a `zte-colab` payload, so the exclusion rules live in the package and are tested there rather
than drifting in a notebook.

In [ ]:
def _mirror(direction: str, date: str, sub: str, local: str | None) -> None:
    """Runs one mirror and reports what moved, or why nothing did."""
    where = ['--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *(('--local', local) if local else ())]
    payload = colab('mirror', *where, '--direction', direction, '--date', date, '--sub', sub)

    if reason := payload['skipped_reason']:
        print(f'nothing mirrored: {reason}')
        return

    print(f'{payload["src"]} -> {payload["dst"]}   ({payload["copied"]} copied, {payload["failed"]} failed)')


def mirror_to_drive(local: str | None = None, sub: str = 'experiments') -> None:
    """Copy the VM's runs to Drive, minus what is rebuildable, so the session survives the machine."""
    _mirror('up', RUN_DATE, sub, local)


def restore_from_drive(run_date: str | None = None, sub: str = 'experiments', local: str | None = None) -> None:
    """Pull a session's runs back to the VM so every `--resume` finds its work after a runtime reset."""
    _mirror('down', run_date or RUN_DATE, sub, local)


def show_resources() -> None:
    """Prints RAM / GPU / disk as they stand, so an out-of-memory kill is predictable rather than a mystery."""
    res = colab('env')['resources']
    gpu = f'{res["gpu"]["name"]} ({res["gpu"]["total_gb"]} GB)' if res['gpu'] else 'none'
    print(f'RAM {res["ram_gb"]} GB · {res["cpu_count"]} cores · {res["free_disk_gb"]} GB free disk · GPU {gpu}')


show_resources()

### 3b · The data — prepare once, and which vantage points are present

`zte-prepare` keys each parallax config by a hash of the fields that actually change the processed bundle and
builds only what the persistent Drive store does not already hold, so a fully-prepared project never touches the
raw `.mat` files again.

**TSR needs the task3 archives** in the Drive dataset folder; ZuCo ships them separately. If the scan below misses
a task, the cell warns loudly and **training (§4) proceeds without that vantage point** — but the archive scan
governs training only. §5 derives its task list from trained checkpoints (VM disk, then the Drive mirror), so an
arm trained in an earlier session is still evaluated even on a runtime where this scan came up short. Drop the
missing archives in and re-run this cell to widen the training matrix to 3 × 3.

In [ ]:
import pathlib

CONFIGS: dict[str, str] = {
    'NR': 'experiments/parallax/parallax_nr.yaml',
    'SR': 'experiments/parallax/parallax_sr.yaml',
    'TSR': 'experiments/parallax/parallax_tsr.yaml',
}

_root = pathlib.Path(DATA_DIR)
_tsr = list(_root.rglob('results*_TSR.mat')) or list(_root.rglob('*task3*'))
TASKS: list[str] = ['NR', 'SR'] + (['TSR'] if _tsr else [])

if len(TASKS) < len(CONFIGS):
    missing = [task for task in CONFIGS if task not in TASKS]
    print('!' * 92)
    print(f'WARNING: archive scan found {len(TASKS)} of {len(CONFIGS)} vantage points — missing: {", ".join(missing)}.')
    print('Training (§4) covers only the tasks listed below; §5 derives its own task list from trained checkpoints.')
    for task in missing:
        # A trained arm on the VM or Drive means the scan missed real archives, not that the task never ran.
        trained = [
            *pathlib.Path(OUT_ROOT).glob(f'parallax_{task.lower()}_*/checkpoints/best.pt'),
            *pathlib.Path(DRIVE_BACKUP).glob(f'parallax_{task.lower()}_*/checkpoints/best.pt'),
        ]
        if trained:
            print(f'checkpoints exist for {task} but its archives were not found — §5 will still evaluate it.')
        else:
            print(f'{task} has no archives and no checkpoints — add its archives to the dataset folder and re-run.')
    print('!' * 92)

print('vantage points:', ', '.join(TASKS))

CONFIG_ARGS: str = ' '.join(f'"{CONFIGS[task]}"' for task in TASKS)

!uv run zte-prepare --root "{DATA_DIR}" --configs {CONFIG_ARGS} \
  --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"

## 4 · The training matrix — this is the multi-hour cell

Three vantage points × three seeds = up to nine runs of the current best-measured recipe (`exp17_base` with a
single-task `dataset.tasks`), each a few hours on a raw-conformer arm. Every run is **resumable**: a finished run
is skipped, an interrupted one continues from its last checkpoint, and each run mirrors to Drive as it completes,
so a reclaimed VM costs at most one epoch. Start it, come back, and re-run the cell verbatim after any
interruption.

A single seed is not a result on this corpus — arms whose only difference was noise have moved between 2 and 9
hits in 700. Three seeds are the floor for an error bar; the comment in the cell names the 5-seed variant.

In [ ]:
HOLDOUT: str = 'ZAB'
# Three seeds are the floor for an error bar; extend to (42, 43, 44, 45, 46) for the 5-seed variant.
SEEDS: tuple[int, ...] = (42, 43, 44)

for task in TASKS:
    for seed in SEEDS:
        name = f'parallax_{task.lower()}_lo{HOLDOUT}_s{seed}'
        print(f'\n=== {name} ' + '=' * 44)
        !uv run zte-run \
          --config "{CONFIGS[task]}" --root "{DATA_DIR}" --name "{name}" --out-root "{OUT_ROOT}" \
          --loso-holdout "{HOLDOUT}" --seed {seed} --data-cache "{PREPARED_LOCAL}" \
          --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume
        mirror_to_drive()

## 5 · The transfer matrix — 3 × 3 × seeds

`zte-parallax transfer` embeds the held-out subject's readings of one task's stimuli with one trained model and
scores closed-set retrieval against that task's gallery: stratified rank percentile with a bootstrap CI, the
length-matched variant beside it, and the menu-capacity audit. Post-processing is fitted on the non-holdout
subjects of the eval task, never on the holdout, and `postprocess_fit` travels in the artifact.

The off-diagonal cells are the point: a model trained on task X, evaluated on task Y, meets a subject it never saw
reading sentences disjoint from everything it was trained on. **Every cell is written straight to Drive** — these
are one-shot JSON/npz artifacts, not streaming checkpoint writes, so the runtime can vanish at any moment without
losing a finished cell — and the loop skips a cell whose `transfer.json` is already on Drive, so this resumes
across runtimes, not just across re-runs. Checkpoints are read from the VM disk when present and from the Drive
mirror otherwise, so a fresh runtime can fill the matrix without retraining anything.

The loop derives its task list **from the checkpoints themselves** — for each task and seed, the VM disk first and
the Drive mirror second — never from §3b's archive scan, so a vantage point whose archives a fresh runtime failed
to detect is still evaluated. A re-run therefore completes the missing cells only: every finished cell is skipped,
and re-running after a partial session fills exactly the gaps it left.

In [ ]:
TRANSFER_ROOT: str = f'{DRIVE_ANALYSIS}/parallax/transfer'


def _best_ckpt(task: str, seed: int) -> pathlib.Path | None:
    """Returns the arm's best checkpoint -- the VM disk first for speed, the Drive mirror for a fresh runtime."""
    run = f'parallax_{task.lower()}_lo{HOLDOUT}_s{seed}'
    candidates = (
        pathlib.Path(OUT_ROOT, run, 'checkpoints', 'best.pt'),
        pathlib.Path(DRIVE_BACKUP, run, 'checkpoints', 'best.pt'),
    )

    return next((p for p in candidates if p.is_file()), None)


# Checkpoints, not §3b's archive scan, decide the matrix: a task trained in any session is evaluated here.
TRANSFER_TASKS: list[str] = [t for t in ('NR', 'SR', 'TSR') if any(_best_ckpt(t, s) for s in SEEDS)]
print('tasks with at least one trained checkpoint:', ', '.join(TRANSFER_TASKS) or 'none')

for train_task in TRANSFER_TASKS:
    for eval_task in TRANSFER_TASKS:
        for seed in SEEDS:
            cell = f'{TRANSFER_ROOT}/{train_task}_to_{eval_task}_s{seed}'
            if pathlib.Path(cell, 'transfer.json').is_file():
                print(f'done, skipping: {train_task} -> {eval_task} · s{seed}')
                continue
            ckpt = _best_ckpt(train_task, seed)
            if ckpt is None:
                print(f'no checkpoint yet (VM or Drive), skipping: parallax_{train_task.lower()}_lo{HOLDOUT}_s{seed}')
                continue
            print(f'\n=== {train_task} -> {eval_task} · s{seed} ' + '=' * 36)
            !uv run zte-parallax transfer \
              --ckpt "{ckpt}" --eval-task {eval_task} --root "{DATA_DIR}" --out "{TRANSFER_ROOT}" \
              --holdout "{HOLDOUT}" --seed {seed}

## 6 · The report and the chamber

`zte-parallax report` aggregates every transfer cell into `PARALLAX.json` (the numbers, per seed), `PARALLAX.md`
(the honest prose reading), and `CHAMBER_DATA.json` (the geometry: each eval task's sentence prototypes reduced to
three dimensions and Procrustes-aligned across the three models' views, so the same sentence can be watched from
all three vantage points).

The report also carries the **menu-decomposition diagnostic** — 2-way accuracy under each scoring rule and length
tolerance, so a gap between the retrieval percentile and the certified menu decomposes into named factors — and it
is a diagnostic only: nothing in it feeds a capacity or a verdict. Beside the stricter prototype capacity it
reports the **enrolled menu capacity**, the best over a sentence's enrolled readings rather than their centroid,
labelled as such.

`zte-parallax chamber` renders that JSON into one self-contained page — no server, no network. The chamber is an
inspection tool, not an audit: the numbers that carry the argument are in `PARALLAX.md`, and the page exists so a
surprising number can be chased into the geometry that produced it.

In [ ]:
PARALLAX_OUT: str = f'{DRIVE_ANALYSIS}/parallax'

!uv run zte-parallax report --transfers "{TRANSFER_ROOT}" --out "{PARALLAX_OUT}"

summary = pathlib.Path(PARALLAX_OUT) / 'PARALLAX.md'
if summary.is_file():
    print(summary.read_text()[:2000])

In [ ]:
import shutil

from IPython.display import IFrame, display

CHAMBER: str = f'{PARALLAX_OUT}/CHAMBER.html'

!uv run zte-parallax chamber --report-dir "{PARALLAX_OUT}" --out "{CHAMBER}"

# Copy to the VM disk before embedding: an iframe reading straight off the Drive mount is slow enough to look broken.
local_chamber = pathlib.Path('res/analysis/CHAMBER.html')
local_chamber.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(CHAMBER, local_chamber)
print(f'{local_chamber}  ({local_chamber.stat().st_size / 1e6:.1f} MB)  ·  also on Drive at {CHAMBER}')
display(IFrame(src=str(local_chamber), width='100%', height=820))

## 7 · How to read the results honestly

Read `PARALLAX.md` in this order, and only in this order.

1. **The diagonal first.** A vantage point evaluated on its own task, on the held-out subject: does a single-task
   encoder reach a stranger's brain at all? A diagonal at chance makes the off-diagonal uninterpretable.
2. **Then the off-diagonal.** Rank percentile with its bootstrap CI, chance at 0.5 — the never-seen subject ×
   never-seen stimuli cell, the strongest generalisation statement this project can make. A CI that includes 0.5
   is a null, and a null is a finding: report it plainly as *no measurable task-invariant transfer at this
   recipe*, not as a failure to be hidden.
3. **Then menu capacity.** The largest exact-length closed set served at the target accuracy — the honest "how
   many sentences could a menu offer" number, with the 5.14-bit length subsidy removed by construction.
4. **Then CKA between the three models.** Convergent geometry without transfer means the vantage points agree
   about structure in a way retrieval cannot yet use; transfer without convergent geometry would be the suspicious
   pattern. Either way, say which one was observed.

What is *not* here: generation (the bit budget prices it out, and no generation number appears in this study), and
pooled `sentence_retrieval` (every number above is the held-out cell). The full method, the falsifiable
predictions and the pre-registered reading of every outcome live in `docs/PARALLAX.md`.

## 8 · The lens — try one reading yourself

The lens walks **one reading** — one subject, one sentence — through a trained checkpoint and shows what the model
did with it: the thought embedding, **word saliency** (each word occluded from the pad mask in turn, scored by the
cosine drop of the re-embedded sentence), **scalp saliency** (electrode groups zeroed by montage region), the
reading's **neighborhood** in the sentence gallery, and — for decoder checkpoints — the **generation trace** with
its null-prefix control beside it.

> inspection, not a result -- no number here is a headline

Every lens artifact carries that sentence verbatim and every page renders it. The lens is a magnifying glass, not
a metric: the honest numbers live in the scoreboard, rebaseline and parallax reports. One thing the page will say
loudly: a subject other than the checkpoint's LOSO holdout is a **training brain**, so what you see there is
memory as much as reach — keep `LENS_SUBJECT` at the holdout unless inspecting memory is the point.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

LENS_OUT: str = f'{DRIVE_ANALYSIS}/lens'

# The first vantage point's first seed by default; point LENS_CKPT at any trained arm to inspect that one instead.
lens_run = f'parallax_{TASKS[0].lower()}_lo{HOLDOUT}_s{SEEDS[0]}'
# The VM checkpoint is preferred for speed; the Drive mirror is what a fresh runtime relies on.
lens_candidates = (
    pathlib.Path(OUT_ROOT, lens_run, 'checkpoints', 'best.pt'),
    pathlib.Path(DRIVE_BACKUP, lens_run, 'checkpoints', 'best.pt'),
)
LENS_CKPT = next((p for p in lens_candidates if p.is_file()), lens_candidates[0])
LENS_SUBJECT: str = HOLDOUT  # any other subject is a training brain, and the page will say so
LENS_INDEX: int = 0  # the LENS_INDEX-th reading by that subject, in the dataset's deterministic order

!uv run zte-lens encode --ckpt "{LENS_CKPT}" --root "{DATA_DIR}" \
  --subject "{LENS_SUBJECT}" --index {LENS_INDEX} --out "{LENS_OUT}" --html

lens_dir = pathlib.Path(LENS_OUT) / f'{lens_run}_{LENS_SUBJECT}_{LENS_INDEX}'
lens = json.loads((lens_dir / 'lens.json').read_text())
reading = lens['reading']

brain = 'held-out subject' if reading['is_holdout'] else 'TRAINING brain'
print(f'{reading["subject"]} ({brain}) · {reading["task"]} · {reading["n_words"]} words')
print(f'text: {reading["text"]}')
print(lens['disclaimer'])

display(pd.DataFrame(lens['neighbors']))

saliency = lens['word_saliency']
fig = go.Figure(
    go.Heatmap(
        z=[saliency['scores']],
        x=reading['words'],
        y=[''],
        colorscale='Reds',
        colorbar={'title': 'cosine drop'},
    )
)
fig.update_layout(title=f'Word saliency ({saliency["method"]}) — inspection, not a result', height=240)
fig.show()

# Copy to the VM disk before embedding: an iframe reading straight off the Drive mount is slow enough to look broken.
local_lens = pathlib.Path('res/analysis/LENS.html')
local_lens.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(lens_dir / 'LENS.html', local_lens)
print(f'{local_lens}  ·  also on Drive at {lens_dir}')
display(IFrame(src=str(local_lens), width='100%', height=820))

In [ ]:
# Any decoder-mode checkpoint works here; the parallax arms are encoder-only, so this points at a decoder run's
# Drive mirror. Edit the session date and run name to a decoder arm you have actually trained.
DECODE_CKPT = pathlib.Path(ZTE_DRIVE, '2026-08-15', 'experiments', 'decode_zte_v2', 'checkpoints', 'best.pt')

if not DECODE_CKPT.is_file():
    print(f'no decoder checkpoint at {DECODE_CKPT} — edit DECODE_CKPT and re-run to trace a generation.')
else:
    !uv run zte-lens decode --ckpt "{DECODE_CKPT}" --root "{DATA_DIR}" \
      --subject "{LENS_SUBJECT}" --index {LENS_INDEX} --out "{LENS_OUT}" --html

    # A run directory is named by its run_name, so the checkpoint's grandparent names the lens directory too.
    decode_dir = pathlib.Path(LENS_OUT) / f'{DECODE_CKPT.parents[1].name}_{LENS_SUBJECT}_{LENS_INDEX}'
    trace = json.loads((decode_dir / 'lens.json').read_text())

    print(f'true sentence : {trace["reading"]["text"]}')
    print(f'generated     : {trace["decode"]["generated"]}')
    print(f'null prefix   : {trace["decode"]["null_prefix_generated"]}')
    print(trace['disclaimer'])

    local_trace = pathlib.Path('res/analysis/LENS_DECODE.html')
    local_trace.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(decode_dir / 'LENS.html', local_trace)
    display(IFrame(src=str(local_trace), width='100%', height=820))

### How to read what you see

- **Saliency is occlusion, not causality.** A high-scoring word is one whose removal moved this model's embedding
  of this reading — a statement about the model, not about the brain, and correlated words share credit
  unpredictably.
- **Neighbors are geometry, not accuracy.** The gallery never contains the query reading itself; the true sentence
  surfacing through *another subject's* reading (flagged `is_true_sentence`) is the interesting event, and its
  absence is not a score.
- **The decode trace is inspection.** Read the generation against its null-prefix control — the bridge's
  *learned* unconditional prefix, the LM with no brain attached (not literal zeros, which would flatter the
  real generation). Fluent text the null prefix also produces came from the language model, not from the EEG.
  The evidence ribbon's diagonal band is a fixed monotonic schedule, not a discovered alignment. The honest
  numbers live in the scoreboard, rebaseline and parallax reports; nothing on a lens page is a headline.

## 9 · The decoder over the winning vantage point (Phase 3)

Phase 3 puts the frozen-LM prefix decoder on top of the best parallax encoder: the bridge trains against the
winner's frozen embedding, and the joint arm then lets the encoder move as well. Two readouts are honest at this
bit budget: **gallery rescoring** — now scored as PMI, the conditional log-likelihood minus the null-prefix one,
so a sentence the LM already finds familiar earns no head start — and **menu capacity** from `zte-rebaseline`.
**Free generation is the pre-registered expected null** (the encoder supplies ~4.7 bits of the ~190 a sentence
needs) and is reported as one; no generated sentence is a headline unless the verdict gate passes. The joint mode
is measurable for the first time: the `best.pt` monitor is stage-aware, resetting at each curriculum boundary, so
the decoder stage's best checkpoint is no longer masked by an earlier stage's loss scale.

Both arms load the winner through `--encoder-ckpt`, and the runs are resumable and mirror to Drive exactly like
§4 — re-run the cells verbatim after any interruption.

In [ ]:
# Pick the best diagonal arm from PARALLAX.md after the §5/§6 re-run; this default is the current best diagonal.
WINNER: str = 'parallax_nr_loZAB_s44'

# The VM checkpoint is preferred for speed; the Drive mirror is what a fresh runtime relies on.
winner_candidates = (
    pathlib.Path(OUT_ROOT, WINNER, 'checkpoints', 'best.pt'),
    pathlib.Path(DRIVE_BACKUP, WINNER, 'checkpoints', 'best.pt'),
)
WINNER_CKPT = next((p for p in winner_candidates if p.is_file()), None)

if WINNER_CKPT is None:
    print(f'no checkpoint for {WINNER} (VM or Drive) — train it in §4, or point WINNER at a trained arm.')
else:
    print(f'encoder for Phase 3: {WINNER_CKPT}')

In [ ]:
DECODER_CONFIGS: tuple[str, ...] = (
    'experiments/decoder/decode_parallax_nr.yaml',
    'experiments/decoder/decode_parallax_nr_joint.yaml',
)
# One seed until the first arm lands and reads honestly; extend to (42, 43, 44) for the error bar.
SEEDS_DECODER: tuple[int, ...] = (42,)

if WINNER_CKPT is None:
    print(f'skipping Phase 3: no encoder checkpoint for {WINNER} — see the cell above.')
else:
    for config in DECODER_CONFIGS:
        for seed in SEEDS_DECODER:
            name = f'{pathlib.Path(config).stem}_lo{HOLDOUT}_s{seed}'
            print(f'\n=== {name} ' + '=' * 44)
            !uv run zte-run \
              --config "{config}" --root "{DATA_DIR}" --name "{name}" --out-root "{OUT_ROOT}" \
              --loso-holdout "{HOLDOUT}" --seed {seed} --encoder-ckpt "{WINNER_CKPT}" \
              --data-cache "{PREPARED_LOCAL}" --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume
            mirror_to_drive()

### How to read the decoder outputs honestly

- **PMI rescoring percentile first**, with its length-stratified cell beside it. The powered readout is
  decoder-rescoring retrieval over the gallery, reported as retrieval, never as generation — and the unstratified
  number alone is not evidence, because sentence length carries 5.14 bits for free.
- **Menu capacity from `zte-rebaseline`** — the largest exact-length closed set served at the target accuracy,
  with the length subsidy removed by construction. It is a diagnostic against any checkpoint and gates nothing.
- **Generation only through the verdict gate.** A generated sentence appears in prose only if
  `verdict['generation_above_controls']` is `True` — honest split, no candidate set, every pre-registered control
  beaten (a skipped or unavailable control fails the clause), permutation $p < 0.05$, prefix-influence KL above
  floor. The expected outcome is a null, and the null is reported plainly.
- **Inspection goes through the lens (§8).** Point `DECODE_CKPT` at a Phase-3 run to watch one generation beside
  its null-prefix control; nothing on a lens page is a headline.